# Car Price Analysis - Machine Learning & Regression Modelling

## Project Overview

This notebook evaluates machine learning approaches for predicting automobile prices based on vehicle specifications and features. We examine linear regression models, regularisation strategies, and dimensionality reduction techniques to address inherent challenges such as feature correlation (multicollinearity) and target skewness.

---

## 1. Theoretical Framework & Definitions

### 1.1 Regression Modelling
**Regression modelling** is a supervised learning methodology used to model and predict continuous numerical targets based on one or more explanatory variables (features). Unlike classification, which predicts discrete categories, regression estimates a function $f(X)$ mapping input features $X$ to continuous output $y$.

### 1.2 Ordinary Least Squares (OLS)
**OLS** stands for **Ordinary Least Squares**. It is the baseline method for linear regression that estimates parameters by minimising the Sum of Squared Residuals (SSR) between actual and predicted target values:

$$\min_{\beta} \sum_{i=1}^{n} \left( y_i - \hat{y}_i \right)^2$$

OLS provides unbiased estimates under Gauss-Markov assumptions, but it is highly sensitive to **multicollinearity** (high correlation between numerical features), which inflates coefficient variances and leads to unstable predictions.

### 1.3 Ridge Regression (L2 Regularisation)
**Ridge Regression** introduces an $L_2$ regularisation penalty on the magnitude of coefficient parameters:

$$\min_{\beta} \left( \sum_{i=1}^{n} \left( y_i - \hat{y}_i \right)^2 + \lambda \sum_{j=1}^{p} \beta_j^2 \right)$$

where $\lambda \ge 0$ controls the penalty strength. By shrinking regression coefficients towards zero (without forcing them to exact zero), Ridge regression penalises large weights, significantly mitigating variance caused by multicollinearity and improving generalisability on unseen test data.

### 1.4 Principal Component Analysis (PCA) & Principal Component Regression (PCR)
* **PCA (Principal Component Analysis):** An unsupervised dimensionality reduction technique that transforms correlated features into an orthogonal set of linearly uncorrelated variables called **Principal Components**. These components sequentially capture maximum variance in the predictor space.
* **PCR (Principal Component Regression):** A two-stage pipeline combining PCA and OLS. First, numerical features undergo PCA transformation to extract the top components capturing a target variance threshold (e.g., 90%). Second, OLS regression is fit on these reduced orthogonal components, eliminating feature correlation before linear modelling.
* **Classification vs. Regression:** While regression models continuous outcomes (such as vehicle dollar price), **classification** predicts discrete categorical outcomes (e.g., classifying a car into 'Budget', 'Mid-Range', or 'Luxury'). PCA can be combined with classification models (e.g., Logistic Regression or SVMs) in a similar pipeline.

---

## 2. Environment Setup & Data Loading

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
sns.set_theme(style="whitegrid", font_scale=1.0)

In [ ]:
# Load dataset
df = pd.read_csv('Car_Price_Cleaned.csv')

---

## 3. Modelling Approach 1: Ridge Regression on Selected Features

### 3.1 Strategy & Preprocessing

Here, we conduct Ridge Regression on a subset of numerical (curbweight, enginesize, horsepower, citympg) and categorical features (drivewheel, carbody, enginelocation). These features were prioritised based on progressive Variance Inflation Factor (VIF) reduction during exploratory data analysis.

We fit models predicting both actual dollar prices and log-transformed prices (np.log1p) to mitigate target distribution skewness.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

# Feature Set & Target Definition
num_features = ['curbweight', 'enginesize', 'horsepower', 'citympg']
cat_features = ['drivewheel', 'carbody', 'enginelocation']

X = df[num_features + cat_features]
y_log = np.log1p(df['price'])  # Log transform price to handle skewness
y_actual = df['price']

# Train-Test Split (Splitting both log target and actual dollar target)
X_train, X_test, y_train_log, y_test_log, y_train_act, y_test_act = train_test_split(
    X, y_log, y_actual, test_size=0.2, random_state=42
)

print("* Train set:", X_train.shape, y_train_log.shape, y_train_act.shape,
      "\n* Test set:",  X_test.shape, y_test_log.shape, y_test_act.shape)

* Train set: (164, 7) (164,) (164,) 
* Test set: (41, 7) (41,) (41,)


In [ ]:
# Check content of train data
X_train.head()

,curbweight,enginesize,horsepower,citympg,drivewheel,carbody,enginelocation
66,2700,134,72,31,rwd,sedan,front
111,3075,120,95,19,rwd,sedan,front
153,2280,92,62,31,fwd,wagon,front
96,1971,97,69,31,fwd,sedan,front
38,2289,110,86,27,fwd,hatchback,front


In [6]:
y_train_act

66     18344.0
111    15580.0
153     6918.0
96      7499.0
38      9095.0
        ...   
106    18399.0
14     24565.0
92      6849.0
179    15998.0
102    14399.0
Name: price, Length: 164, dtype: float64

In [7]:
y_train_log

66      9.817112
111     9.653808
153     8.842027
96      8.922658
38      9.115590
         ...    
106     9.820106
14     10.109119
92      8.832004
179     9.680281
102     9.574983
Name: price, Length: 164, dtype: float64

### 3.2 Evaluation Metrics (Dollar Scale vs. Log Scale)

In [ ]:
# Preprocessing Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first'), cat_features)
    ]
)

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

# Fit RidgeCV Model
alphas = np.logspace(-3, 3, 100)
ridge_model = RidgeCV(alphas=alphas)
ridge_model.fit(X_train_prep, y_train_log)

# Predict on Test Set
y_pred_log = ridge_model.predict(X_test_prep)
y_pred_dollar = np.expm1(y_pred_log)  # Exponentiate back to actual dollar values

# Calculate Detailed Performance Metrics
n = X_test.shape[0]          # Number of test observations
p = X_train_prep.shape[1]    # Number of features after One-Hot Encoding

print("Number of test observations:", n,"\nNumer of features after One-Hot Encoding:",p)

Number of test observations: 41 
Numer of features after One-Hot Encoding: 11


In [9]:
y_pred_log

array([10.11018174,  9.73895284,  9.20154183,  9.3951182 , 10.00254871,
        8.82110655,  8.84485971,  9.03468991,  9.11496912,  8.95139016,
        9.53217701,  8.99917667,  9.61253886,  9.14170539, 10.51599082,
        8.79739453,  8.2724234 ,  9.60000481,  9.26291575,  9.22842822,
        9.24417217,  9.89986111,  8.87283741,  8.39822541,  8.9082493 ,
       10.19506229,  9.41801005,  9.56504179,  8.74690525,  9.68669881,
       10.06026238,  8.78440108,  8.86443495,  9.85039122,  9.05006757,
       10.00059417,  9.33103746,  9.31830029,  8.78257348,  9.65263076,
        8.95688172])

In [10]:
y_pred_dollar

array([24591.12971129, 16964.76612688,  9911.40047728, 12028.51151162,
       22081.67636421,  6774.75820669,  6937.63055509,  8388.11165869,
        9089.35393847,  7717.61454035, 13795.59374282,  8095.41517581,
       14950.0806883 ,  9335.67419249, 36899.8852216 ,  6615.98117883,
        3913.42373489, 14763.8526465 , 10538.8200939 , 10181.52420977,
       10343.10604438, 19926.60247592,  7134.49863988,  4438.18202638,
        7391.70732547, 26769.67338508, 12307.06545694, 14256.54879469,
        6290.18832325, 16101.00064558, 23393.64382525,  6530.55990955,
        7074.79402398, 18964.77308118,  8518.11355646, 22038.55712165,
       11281.83095551, 11139.03093409,  6518.63377393, 15561.67589197,
        7760.11835296])

In [11]:
# 1. Metrics on Original Dollar Scale
r2 = r2_score(y_test_act, y_pred_dollar)
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
mae = mean_absolute_error(y_test_act, y_pred_dollar)
rmse = np.sqrt(mean_squared_error(y_test_act, y_pred_dollar))
mape = mean_absolute_percentage_error(y_test_act, y_pred_dollar) * 100

# Display Results
print("=" * 45)
print(" RIDGE REGRESSION MODEL PERFORMANCE (Dollars)  ")
print("=" * 45)
print(f"Optimal Alpha Penalty : {ridge_model.alpha_:.4f}")
print("-" * 45)
print(f"R² Score              : {r2:.4f}")
print(f"Adjusted R² Score     : {adj_r2:.4f}")
print(f"Mean Absolute Error   : ${mae:,.2f}")
print(f"Root Mean Sq. Error   : ${rmse:,.2f}")
print(f"Mean Abs. % Error     : {mape:.2f}%")
print("=" * 45)

 RIDGE REGRESSION MODEL PERFORMANCE (Dollars)  
Optimal Alpha Penalty : 0.3511
---------------------------------------------
R² Score              : 0.8576
Adjusted R² Score     : 0.8036
Mean Absolute Error   : $2,054.84
Root Mean Sq. Error   : $3,352.62
Mean Abs. % Error     : 13.64%


In [12]:
# 2. Metrics on Log Price Scale
r2 = r2_score(y_test_log, y_pred_log)
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
mae = mean_absolute_error(y_test_log, y_pred_log)
rmse = np.sqrt(mean_squared_error(y_test_log, y_pred_log))
mape = mean_absolute_percentage_error(y_test_log, y_pred_log) * 100

# Display Results
print("=" * 45)
print("   RIDGE REGRESSION MODEL PERFORMANCE (Log)  ")
print("=" * 45)
print(f"Optimal Alpha Penalty : {ridge_model.alpha_:.4f}")
print("-" * 45)
print(f"R² Score              : {r2:.4f}")
print(f"Adjusted R² Score     : {adj_r2:.4f}")
print(f"Mean Absolute Error   : {mae:,.2f}")
print(f"Root Mean Sq. Error   : {rmse:,.2f}")
print(f"Mean Abs. % Error     : {mape:.2f}%")
print("=" * 45)

   RIDGE REGRESSION MODEL PERFORMANCE (Log)  
Optimal Alpha Penalty : 0.3511
---------------------------------------------
R² Score              : 0.8806
Adjusted R² Score     : 0.8353
Mean Absolute Error   : 0.14
Root Mean Sq. Error   : 0.18
Mean Abs. % Error     : 1.50%


Observation: The regression model predicting log prices slightly outperforms the model fitted on raw dollar values. The log-target model achieves an $R^2$ of 0.88 versus 0.85 on raw dollars. Furthermore, log transformation significantly stabilises high-end price prediction error variance, dropping Mean Absolute Percentage Error (MAPE) substantially.

In [13]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [14]:
df_clean = df.copy()

In [15]:
# Define Feature Sets
num_cols = ['enginesize', 'horsepower', 'curbweight', 'carwidth', 'carlength', 'wheelbase', 'boreratio', 'citympg', 'highwaympg', 'cylindernumber', 'doornumber']
cat_cols = ['manufacturer', 'carbody', 'drivewheel', 'enginelocation', 'fueltype', 'aspiration', 'fuelsystem']

In [16]:
X = df_clean[num_cols + cat_cols]
y = df_clean['price']
y_log = np.log(df_clean['price'])  # Log-transform target to handle skewness

In [17]:
# Train / Test Split
X_train, X_test, y_train, y_test, y_train_log, y_test_log, = train_test_split(X, y, y_log, test_size=0.2, random_state=42)

In [18]:
# -------------------------------------------------------------------
# 2. Define Shared Preprocessing Transformers
# -------------------------------------------------------------------
preprocessor_standard = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
])

preprocessor_pca = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=0.90))  # Retain 90% of numerical feature variance
    ]), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
])

In [19]:
# -------------------------------------------------------------------
# 3. Build Comparison Pipelines
# -------------------------------------------------------------------
pipelines = {
    'OLS (Baseline)': Pipeline([('prep', preprocessor_standard), ('reg', LinearRegression())]),
    'PCR (PCA + OLS)': Pipeline([('prep', preprocessor_pca), ('reg', LinearRegression())]),
    'Ridge (L2 Penalty)': Pipeline([('prep', preprocessor_standard), ('reg', Ridge(alpha=1.0))])
}

In [20]:
# -------------------------------------------------------------------
#    Cross-Validation & Test Set Evaluation Loop (Dollars)
# -------------------------------------------------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = []

print("=== 5-FOLD CROSS-VALIDATION RESULTS (LOG SCALE R²) ===")
for name, pipe in pipelines.items():
    # 5-Fold Cross Validation on Training Data
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=kf, scoring='r2')
    
    # Fit model on training set and evaluate on test set
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    # Compute metrics in original currency
    test_r2 = r2_score(y_test, y_pred)
    test_mae = mean_absolute_error(y_test, y_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    results.append({
        'Model Pipeline': name,
        'Mean CV R²': f"{cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})",
        'Test R² ($)': round(test_r2, 4),
        'Test MAE ($)': f"{test_mae:,.2f}",
        'Test RMSE ($)': f"{test_rmse:,.2f}"
    })

=== 5-FOLD CROSS-VALIDATION RESULTS (LOG SCALE R²) ===


/Users/mariaoanacucu/Documents/VS Code Projects/Car_feature_price_analysis/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:228: UserWarning: Found unknown categories in columns [0, 6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/mariaoanacucu/Documents/VS Code Projects/Car_feature_price_analysis/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:228: UserWarning: Found unknown categories in columns [6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/mariaoanacucu/Documents/VS Code Projects/Car_feature_price_analysis/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:228: UserWarning: Found unknown categories in columns [0, 6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/mariaoanacucu/Documents/VS Code Projects/Car_feature_price_analysis/.venv/lib/python3.12/site-packages/sklearn/

In [21]:
# Output Summary Comparison Table
results_df = pd.DataFrame(results)
results_df

,Model Pipeline,Mean CV R²,Test R² ($),Test MAE ($),Test RMSE ($)
0,OLS (Baseline),0.8818 (+/- 0.0441),0.8812,"2,019.68","3,061.84"
1,PCR (PCA + OLS),0.9054 (+/- 0.0457),0.8658,"2,103.38","3,255.15"
2,Ridge (L2 Penalty),0.8934 (+/- 0.0428),0.8918,"2,055.48","2,922.08"


In [22]:
# -------------------------------------------------------------------
#    Cross-Validation & Test Set Evaluation Loop (Log)
# -------------------------------------------------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = []

print("=== 5-FOLD CROSS-VALIDATION RESULTS (LOG SCALE R²) ===")
for name, pipe in pipelines.items():
    # 5-Fold Cross Validation on Training Data
    cv_scores = cross_val_score(pipe, X_train, y_train_log, cv=kf, scoring='r2')
    
    # Fit model on training set and evaluate on test set
    pipe.fit(X_train, y_train_log)
    y_pred_log = pipe.predict(X_test)
    
    #Convert predictions back to original Dollar Values ($)
    y_test_dollars = np.exp(y_test_log)
    y_pred_dollars = np.exp(y_pred_log)
    
    # Compute metrics in original currency
    test_r2 = r2_score(y_test_dollars, y_pred_dollars)
    test_mae = mean_absolute_error(y_test_dollars, y_pred_dollars)
    test_rmse = np.sqrt(mean_squared_error(y_test_dollars, y_pred_dollars))

    
    results.append({
        'Model Pipeline': name,
        'Mean CV R²': f"{cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})",
        'Test R² ($)': round(test_r2, 4),
        'Test MAE ($)': f"{test_mae:,.2f}",
        'Test RMSE ($)': f"{test_rmse:,.2f}"
    })

=== 5-FOLD CROSS-VALIDATION RESULTS (LOG SCALE R²) ===


/Users/mariaoanacucu/Documents/VS Code Projects/Car_feature_price_analysis/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:228: UserWarning: Found unknown categories in columns [0, 6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/mariaoanacucu/Documents/VS Code Projects/Car_feature_price_analysis/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:228: UserWarning: Found unknown categories in columns [6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/mariaoanacucu/Documents/VS Code Projects/Car_feature_price_analysis/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:228: UserWarning: Found unknown categories in columns [0, 6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/mariaoanacucu/Documents/VS Code Projects/Car_feature_price_analysis/.venv/lib/python3.12/site-packages/sklearn/

In [23]:
results_df = pd.DataFrame(results)
results_df

,Model Pipeline,Mean CV R²,Test R² ($),Test MAE ($),Test RMSE ($)
0,OLS (Baseline),0.8882 (+/- 0.0270),0.9277,"1,634.30","2,388.36"
1,PCR (PCA + OLS),0.9083 (+/- 0.0269),0.9098,"1,683.89","2,669.15"
2,Ridge (L2 Penalty),0.9070 (+/- 0.0269),0.9421,"1,470.40","2,137.85"


The ridge regression using all features, and predicting the log price, outperforms all models in this comparison, and its R² score is higher even than that of the initial ridge regression predicting log price. This brief model comparison showed that combinining highly correlated variables was not needed, and that the ridge regression combining all models showed the best performance, although the risk of overfitting exists. To account for this, feature extraction using hyperparameter optimisation, as well as precise model performance indicators, can illustrate a more detailed picture of each model's suitability. However, the author maintains that this dataset is too small for such detailed research to be performed, and might be outside the scope of the present business requirements.